# 🔍 Classification des Défaillances de Contrôle Interne par Type

---

**Objectif :** Analyser, classifier et prédire les défaillances de contrôle interne à l'aide de techniques de machine learning.

**Auteur :** [Votre Nom]  
**Date :** Mars 2026

---

### 📋 Table des Matières
1. [Installation & Imports](#1)
2. [Génération du Dataset](#2)
3. [Analyse Exploratoire](#3)
4. [Graphiques G1–G4 (Analyse)](#4)
5. [Préparation & Modélisation](#5)
6. [Graphiques G5–G8 (Modèle)](#6)
7. [Rapport Final](#7)

## Cellule 0 — Installation des Dépendances <a id='1'></a>

In [ ]:
# ─── Cellule 0 : Installation ───────────────────────────────────────────────
!pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn -q
print('✅ Installation terminée')

## Cellule 1 — Imports et Configuration <a id='1b'></a>

In [ ]:
# ─── Cellule 1 : Imports & Configuration ────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, roc_auc_score
)
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

# Configuration graphique globale
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

# Palette de couleurs cohérente
COLORS = ['#2E75B6', '#70AD47', '#FFC000', '#FF4444']
PALETTE = {'Faible': '#70AD47', 'Moyen': '#FFC000', 'Élevé': '#FF8C00', 'Critique': '#FF4444'}

print('✅ Bibliothèques chargées avec succès')

## Cellule 2 — Génération du Dataset Synthétique <a id='2'></a>

In [ ]:
# ─── Cellule 2 : Génération du Dataset ───────────────────────────────────────
np.random.seed(42)
n = 500

TYPES    = ['Préventif', 'Détectif', 'Correctif', 'Directif']
DEPTS    = ['Finance', 'IT', 'RH', 'Opérations', 'Conformité']
RISQUES  = ['Faible', 'Moyen', 'Élevé', 'Critique']

# Distribution réaliste des types de contrôle
probs_type = [0.35, 0.28, 0.22, 0.15]

df = pd.DataFrame({
    'type_controle'    : np.random.choice(TYPES, n, p=probs_type),
    'departement'      : np.random.choice(DEPTS, n,
                             p=[0.30, 0.25, 0.18, 0.15, 0.12]),  # Finance domine
    'cout_estime'      : np.abs(np.random.normal(47, 22, n)).round(1),
    'duree_resolution' : np.abs(np.random.normal(18, 12, n)).astype(int) + 1,
    'frequence'        : np.random.randint(1, 13, n),
    'mois'             : np.random.choice(range(1, 13), n,
                             p=[0.06,0.06,0.11,0.07,0.07,0.07,
                                0.07,0.07,0.11,0.08,0.08,0.15]),  # Pics mars/sep/déc
})

# Assignation réaliste du niveau de risque basée sur un score composite
def assign_risk(row):
    score = (
        (row['cout_estime']      / 150) * 0.40 +
        (row['frequence']        / 12)  * 0.35 +
        (row['duree_resolution'] / 90)  * 0.25
    )
    # Bonus risque Finance & IT
    if row['departement'] in ['Finance', 'IT']:
        score *= 1.15
    if   score < 0.25: return 'Faible'
    elif score < 0.50: return 'Moyen'
    elif score < 0.75: return 'Élevé'
    else:              return 'Critique'

df['niveau_risque'] = df.apply(assign_risk, axis=1)

# Aperçu
print('=' * 60)
print('APERÇU DU DATASET')
print('=' * 60)
print(df.head(10).to_string(index=False))
print(f'\n📊 Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes')
print('\n📈 Distribution des niveaux de risque :')
print(df['niveau_risque'].value_counts(normalize=True).round(3).to_string())

## Cellule 3 — Analyse Exploratoire (EDA) <a id='3'></a>

In [ ]:
# ─── Cellule 3 : Statistiques Descriptives ───────────────────────────────────
print('=' * 60)
print('STATISTIQUES DESCRIPTIVES')
print('=' * 60)
print(df.describe().round(2).to_string())

print('\n' + '=' * 60)
print('DISTRIBUTION PAR TYPE DE CONTRÔLE')
print('=' * 60)
print(df['type_controle'].value_counts().to_string())

print('\n' + '=' * 60)
print('VALEURS MANQUANTES')
print('=' * 60)
print(df.isnull().sum().to_string())

print('\n✅ Aucune valeur manquante détectée')

## Cellule 4 — Graphiques G1, G2, G3, G4 <a id='4'></a>

In [ ]:
# ─── Cellule 4 : Visualisations Analytiques ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Analyse des Défaillances de Contrôle Interne',
             fontsize=16, fontweight='bold', y=1.01)

# ── G1 : Pie chart – Distribution par Type ──────────────────────────────────
counts = df['type_controle'].value_counts()
wedges, texts, autotexts = axes[0, 0].pie(
    counts,
    labels=counts.index,
    autopct='%1.1f%%',
    colors=COLORS,
    startangle=90,
    explode=[0.05] * 4,
    shadow=True,
    textprops={'fontsize': 10}
)
for at in autotexts:
    at.set_fontweight('bold')
axes[0, 0].set_title('G1 – Distribution par Type de Contrôle',
                      fontweight='bold', pad=15)

# ── G2 : Heatmap – Département × Niveau de Risque ───────────────────────────
pivot = pd.crosstab(df['departement'], df['niveau_risque'])
pivot = pivot[['Faible', 'Moyen', 'Élevé', 'Critique']]
sns.heatmap(
    pivot, annot=True, fmt='d',
    cmap='YlOrRd', ax=axes[0, 1],
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Nombre de défaillances'}
)
axes[0, 1].set_title('G2 – Niveau de Risque par Département',
                      fontweight='bold')
axes[0, 1].set_xlabel('Niveau de Risque')
axes[0, 1].set_ylabel('Département')

# ── G3 : Courbe – Évolution mensuelle ───────────────────────────────────────
monthly = df.groupby('mois').size().reset_index(name='count')
MOIS_LABELS = ['Jan','Fév','Mar','Avr','Mai','Jun',
               'Jul','Aoû','Sep','Oct','Nov','Déc']
axes[1, 0].plot(
    monthly['mois'], monthly['count'],
    marker='o', color='#2E75B6', linewidth=2.5, markersize=7
)
axes[1, 0].fill_between(
    monthly['mois'], monthly['count'],
    alpha=0.15, color='#2E75B6'
)
# Annoter pics
for _, row in monthly.iterrows():
    if row['count'] >= monthly['count'].quantile(0.80):
        axes[1, 0].annotate(
            str(int(row['count'])),
            xy=(row['mois'], row['count']),
            xytext=(0, 8), textcoords='offset points',
            ha='center', fontsize=9, color='#2E75B6', fontweight='bold'
        )
axes[1, 0].set_xticks(range(1, 13))
axes[1, 0].set_xticklabels(MOIS_LABELS, rotation=45)
axes[1, 0].set_xlabel('Mois')
axes[1, 0].set_ylabel('Nombre de Défaillances')
axes[1, 0].set_title('G3 – Évolution Mensuelle des Défaillances',
                      fontweight='bold')

# ── G4 : Barplot horizontal – Coût moyen par type ───────────────────────────
cout_stats = df.groupby('type_controle')['cout_estime'].agg(['mean', 'sem']).reset_index()
cout_stats = cout_stats.sort_values('mean')
bars = axes[1, 1].barh(
    cout_stats['type_controle'],
    cout_stats['mean'],
    xerr=cout_stats['sem'] * 1.96,
    color=COLORS[:len(cout_stats)],
    edgecolor='white', height=0.6,
    error_kw={'ecolor': 'gray', 'capsize': 4}
)
for bar, val in zip(bars, cout_stats['mean']):
    axes[1, 1].text(
        bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
        f'{val:.1f} kMAD', va='center', fontsize=9
    )
axes[1, 1].set_xlabel('Coût Moyen Estimé (kMAD)')
axes[1, 1].set_title('G4 – Coût Moyen par Type de Contrôle (IC 95%)',
                      fontweight='bold')

plt.tight_layout()
plt.savefig('graphiques_analyse.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Graphiques G1-G4 sauvegardés → graphiques_analyse.png')

## Cellule 5 — Préparation des Données & Modélisation <a id='5'></a>

In [ ]:
# ─── Cellule 5 : Encodage & Entraînement des Modèles ────────────────────────
df_enc = df.copy()
le = LabelEncoder()

# Encodage des variables catégorielles
for col in ['type_controle', 'departement', 'niveau_risque']:
    df_enc[col] = le.fit_transform(df[col])

# Mapping pour l'affichage
le_risque = LabelEncoder().fit(df['niveau_risque'])
LABELS = list(le_risque.classes_)  # ['Critique','Élevé','Faible','Moyen']

# Features & Target
FEATURES = ['type_controle', 'departement', 'cout_estime',
            'duree_resolution', 'frequence', 'mois']
X = df_enc[FEATURES]
y = le_risque.transform(df['niveau_risque'])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# ── Comparaison des Modèles ──────────────────────────────────────────────────
modeles = {
    'Random Forest'       : RandomForestClassifier(n_estimators=200, max_depth=10,
                                                    min_samples_split=5,
                                                    class_weight='balanced', random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=150, max_depth=5,
                                                        learning_rate=0.1, random_state=42),
    'Régression Logistique': LogisticRegression(max_iter=1000, class_weight='balanced',
                                                 random_state=42)
}

resultats = []
for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    acc  = accuracy_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred, average='weighted')
    cv   = cross_val_score(modele, X, y, cv=5, scoring='accuracy').mean()
    resultats.append({'Modèle': nom, 'Précision Test': f'{acc:.2%}',
                      'F1-Score': f'{f1:.3f}', 'CV Accuracy': f'{cv:.2%}'})
    print(f'✓ {nom}: Acc={acc:.2%}, F1={f1:.3f}, CV={cv:.2%}')

df_res = pd.DataFrame(resultats)
print('\n' + '=' * 60)
print('COMPARAISON DES MODÈLES')
print('=' * 60)
print(df_res.to_string(index=False))

# Modèle retenu : Random Forest
rf = modeles['Random Forest']
y_pred_rf = rf.predict(X_test)
print(f'\n🏆 Modèle retenu : Random Forest ({accuracy_score(y_test, y_pred_rf):.2%})')

## Cellule 6 — Graphiques G5, G6, G7, G8 <a id='6'></a>

In [ ]:
# ─── Cellule 6 : Visualisations Modèle & Corrélations ───────────────────────
fig2, axes2 = plt.subplots(2, 2, figsize=(16, 12))
fig2.suptitle('Modélisation & Analyse des Corrélations',
              fontsize=16, fontweight='bold', y=1.01)

# ── G5 : Matrice de Corrélation ──────────────────────────────────────────────
corr_vars = ['type_controle', 'departement', 'cout_estime',
             'duree_resolution', 'frequence', 'niveau_risque']
corr = df_enc[corr_vars].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(
    corr, annot=True, fmt='.2f',
    cmap='coolwarm', ax=axes2[0, 0],
    center=0, square=True, linewidths=0.5,
    cbar_kws={'shrink': 0.8}
)
axes2[0, 0].set_title('G5 – Matrice de Corrélation', fontweight='bold')
axes2[0, 0].tick_params(axis='x', rotation=45)

# ── G6 : Feature Importance ──────────────────────────────────────────────────
feat_imp = pd.Series(
    rf.feature_importances_, index=FEATURES
).sort_values()
colors_fi = ['#2E75B6' if v >= feat_imp.median() else '#A9C4DE'
             for v in feat_imp.values]
feat_imp.plot(
    kind='barh', ax=axes2[0, 1],
    color=colors_fi, edgecolor='white'
)
for i, (idx, val) in enumerate(feat_imp.items()):
    axes2[0, 1].text(
        val + 0.002, i,
        f'{val:.1%}', va='center', fontsize=9
    )
axes2[0, 1].set_xlabel('Importance (%)')
axes2[0, 1].set_title('G6 – Importance des Variables (Random Forest)',
                       fontweight='bold')

# ── G7 : Matrice de Confusion ────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred_rf)
cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
annot = np.array(
    [[f'{cm[i,j]}\n({cm_pct[i,j]:.0f}%)'
      for j in range(cm.shape[1])]
     for i in range(cm.shape[0])]
)
sns.heatmap(
    cm, annot=annot, fmt='',
    cmap='Blues', ax=axes2[1, 0],
    xticklabels=LABELS, yticklabels=LABELS,
    linewidths=0.5, linecolor='white'
)
axes2[1, 0].set_xlabel('Prédit')
axes2[1, 0].set_ylabel('Réel')
axes2[1, 0].set_title(
    f'G7 – Matrice de Confusion (Acc: {accuracy_score(y_test, y_pred_rf):.1%})',
    fontweight='bold'
)

# ── G8 : Distribution des Coûts par Type (KDE) ──────────────────────────────
for t, c in zip(TYPES, COLORS):
    subset = df[df['type_controle'] == t]['cout_estime']
    subset.plot(
        kind='kde', ax=axes2[1, 1],
        label=t, color=c, linewidth=2.5
    )
    axes2[1, 1].axvline(
        subset.mean(), color=c,
        linestyle='--', linewidth=1, alpha=0.7
    )
axes2[1, 1].set_xlabel('Coût Estimé (kMAD)')
axes2[1, 1].set_ylabel('Densité')
axes2[1, 1].set_title('G8 – Distribution des Coûts par Type (lignes = moyennes)',
                       fontweight='bold')
axes2[1, 1].legend()

plt.tight_layout()
plt.savefig('graphiques_modelisation.png', dpi=300, bbox_inches='tight')
plt.show()
print('✅ Graphiques G5-G8 sauvegardés → graphiques_modelisation.png')

## Cellule 7 — Rapport de Classification Complet <a id='7'></a>

In [ ]:
# ─── Cellule 7 : Rapport Final ───────────────────────────────────────────────
print('=' * 70)
print('        RAPPORT DE CLASSIFICATION — CONTRÔLE INTERNE')
print('=' * 70)

print('\n📊 DISTRIBUTION DES TYPES DE CONTRÔLE :')
for t, c in df['type_controle'].value_counts(normalize=True).items():
    bar = '█' * int(c * 40)
    print(f'  {t:<15} {bar:<42} {c:.1%}')

print('\n🏢 DÉFAILLANCES CRITIQUES PAR DÉPARTEMENT :')
crit = df[df['niveau_risque']=='Critique']['departement'].value_counts()
for dept, nb in crit.items():
    print(f'  {dept:<15} : {nb} défaillances critiques')

print('\n💰 COÛT MOYEN PAR TYPE DE CONTRÔLE :')
cout_type = df.groupby('type_controle')['cout_estime'].mean().sort_values(ascending=False)
for t, c in cout_type.items():
    print(f'  {t:<15} : {c:>6.1f} kMAD')

print('\n🤖 PERFORMANCE DU MODÈLE RANDOM FOREST :')
print(classification_report(y_test, y_pred_rf, target_names=LABELS))

print('\n💡 RECOMMANDATIONS :')
recs = [
    '1. Renforcer les contrôles préventifs (Finance & IT en priorité)',
    '2. Déployer le modèle RF pour le scoring automatique des défaillances',
    '3. Planifier des audits ciblés en mars, septembre et décembre',
    '4. Réviser les politiques de séparation des tâches',
    '5. Mettre en place un tableau de bord de suivi en temps réel',
]
for r in recs:
    print(f'  {r}')

print('\n' + '=' * 70)
print('✅ Analyse terminée. Fichiers sauvegardés :')
print('   • graphiques_analyse.png')
print('   • graphiques_modelisation.png')
print('=' * 70)

---
### 📁 Téléchargement des Graphiques depuis Colab

```python
# Exécuter pour télécharger les fichiers générés
from google.colab import files
files.download('graphiques_analyse.png')
files.download('graphiques_modelisation.png')
```

---
*Compte rendu généré automatiquement — Classification des Défaillances de Contrôle Interne — Mars 2026*